In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
try:
    ROOT = Path(__file__).resolve().parent   # script na raiz do projeto
except NameError:
    ROOT = Path.cwd()                        # notebook aberto na raiz do projeto
 
SRC = ROOT / "dados" / "silver" / "history_silver.csv"
DST = ROOT / "dados" / "gold" / "history_gold.csv"
DST.parent.mkdir(parents=True, exist_ok=True)
 
df = pd.read_csv(SRC)
 
# ---------- 1. Limpeza ----------
df["Servico_Telefonico"] = df["Servico_Telefonico"].str.strip().str.capitalize()   # yes/no -> Yes/No
yn = lambda s: (s == "Yes").astype("int8")   # "No internet service"/"No phone service" -> 0
 
# ---------- 2. Gold: colunas mantidas / criadas ----------
g = pd.DataFrame()
g["ID_Cliente"] = df["ID_Cliente"].astype("string")
 
# Perfil do cliente
g["Cliente_Idoso"] = df["Cliente_Idoso"].astype("int8")
g["Parceiro"] = yn(df["Parceiro"])
g["Dependentes"] = yn(df["Dependentes"])
g["Tem_Familia"] = ((g["Parceiro"] + g["Dependentes"]) > 0).astype("int8")
 
# Tempo de relacionamento
g["Tempo_Permanencia_Meses"] = df["Tempo_Permanencia(MESES)"].astype("int16")
g["Faixa_Permanencia"] = pd.cut(
    g["Tempo_Permanencia_Meses"], bins=[-1, 6, 12, 24, 48, 1000],
    labels=["0-6", "7-12", "13-24", "25-48", "48+"]).astype("string")
 
# Produto contratado
g["Servico_Internet"] = df["Servico_Internet"].astype("string")      # DSL / Fiber optic / No
g["Seguranca_Online"] = yn(df["Seguranca_Online"])
g["Backup_Online"] = yn(df["Backup_Online"])
g["Protecao_Dispositivo"] = yn(df["Protecao_Dispositivo"])
g["Suporte_Tecnico"] = yn(df["Suporte_Tecnico"])
g["Qtd_Servicos_Protecao"] = (g[["Seguranca_Online", "Backup_Online",
                                 "Protecao_Dispositivo", "Suporte_Tecnico"]].sum(axis=1)).astype("int8")
g["Qtd_Servicos_Total"] = (
    yn(df["Servico_Telefonico"]) + yn(df["Multiplas_Linhas"]) + (df["Servico_Internet"] != "No").astype("int8")
    + g["Qtd_Servicos_Protecao"] + yn(df["Streaming_TV"]) + yn(df["Streaming_Filmes"])
).astype("int8")
 
# Contrato e pagamento
g["Contrato"] = df["Contrato"].astype("string")
g["Metodo_Pagamento"] = df["Metodo_Pagamento"].astype("string")
g["Pagamento_Automatico"] = df["Metodo_Pagamento"].str.contains("automatic").astype("int8")
g["Faturamento_Sem_Papel"] = yn(df["Faturamento_Sem_Papel"])
 
# Financeiro
g["Cobranca_Mensal"] = df["Cobranca_Mensal"].astype("float32")
g["Cobranca_Por_Servico"] = (df["Cobranca_Mensal"] / g["Qtd_Servicos_Total"].clip(lower=1)).round(2).astype("float32")
media_hist = np.where(g["Tempo_Permanencia_Meses"] > 0,
                      df["Cobranca_Total"] / g["Tempo_Permanencia_Meses"].clip(lower=1), np.nan)
g["Variacao_Preco_Pct"] = ((df["Cobranca_Mensal"] / media_hist) - 1).round(3).astype("float32")
# sinaliza linhas em que Cobranca_Total foge muito de Permanencia x Cobranca_Mensal
esperado = df["Cobranca_Mensal"] * g["Tempo_Permanencia_Meses"]
g["Flag_Cobranca_Inconsistente"] = ((df["Cobranca_Total"] - esperado).abs() > 0.5 * esperado.clip(lower=100)).astype("int8")
 
# Segmento de risco combinado
g["Perfil_Alto_Risco"] = (
    (df["Contrato"] == "Month-to-month") & (df["Metodo_Pagamento"] == "Electronic check")
    & (df["Servico_Internet"] == "Fiber optic") & (df["Suporte_Tecnico"] != "Yes")
).astype("int8")
 
# Alvo
g["Cancelou"] = (df["Cancelamento"] == "Yes").astype("int8")
 
# ---------- 3. Salvar ----------
g.to_csv(DST, index=False)
print(g.shape)
print(g.dtypes)
print(g.isna().sum()[lambda s: s > 0])
print("Perfil_Alto_Risco:", g.Perfil_Alto_Risco.sum(), "clientes | churn =",
      g.loc[g.Perfil_Alto_Risco == 1, "Cancelou"].mean().round(3), "vs resto", g.loc[g.Perfil_Alto_Risco == 0, "Cancelou"].mean().round(3))
print("Flag inconsistente:", g.Flag_Cobranca_Inconsistente.sum())
print(g.groupby("Qtd_Servicos_Protecao").Cancelou.agg(["mean", "size"]).round(3))
print(g.Variacao_Preco_Pct.describe().round(2))
print(g.head(3).T)

(5634, 24)
ID_Cliente                     string[python]
Cliente_Idoso                            int8
Parceiro                                 int8
Dependentes                              int8
Tem_Familia                              int8
Tempo_Permanencia_Meses                 int16
Faixa_Permanencia              string[python]
Servico_Internet               string[python]
Seguranca_Online                         int8
Backup_Online                            int8
Protecao_Dispositivo                     int8
Suporte_Tecnico                          int8
Qtd_Servicos_Protecao                    int8
Qtd_Servicos_Total                       int8
Contrato                       string[python]
Metodo_Pagamento               string[python]
Pagamento_Automatico                     int8
Faturamento_Sem_Papel                    int8
Cobranca_Mensal                       float32
Cobranca_Por_Servico                  float32
Variacao_Preco_Pct                    float32
Flag_Cobranca_Inconsist